# exp03 — Model Organism: LoRA Acrostic Fine-tuning

**Goal:** eliminate the input confound by fine-tuning Llama-3.1-8B (LoRA) to encode acrostic
**without explicit instruction**. Both open and stego conditions receive identical prompts.

**Design (Fang & Marks 2512.01222):**
- Train: (open_prompt → stego_CoT) pairs from exp01 valid_pairs.json
- Inference: base model vs LoRA model, identical prompts
- Expected: layer-0 AUROC drops to ~0.5, signal peaks at mid/late layers (28–32)

**Hardware:** Colab Pro A100 (40 GB VRAM)


In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REAL_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'

if IN_COLAB:
    if not os.path.exists('stego_CoT'):
        os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
    os.chdir('stego_CoT')
    os.system('git pull -q origin main')

    # Install packages BEFORE any ML imports — no restart needed on fresh runtime.
    # torchao must be >=0.16.0; install it here so peft sees the correct version at import time.
    print('Installing / verifying packages...')
    ret = os.system(f'{sys.executable} -m pip install -q "torchao>=0.16.0" -r requirements.txt')
    print(f'pip exit code: {ret}')

    from google.colab import userdata, drive
    drive.mount('/content/drive')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    INPUT_DIR  = '/content/drive/MyDrive/stego_CoT/exp01'
    OUTPUT_DIR = '/content/drive/MyDrive/stego_CoT/exp03'
else:
    INPUT_DIR  = 'results/exp01'
    OUTPUT_DIR = 'results/exp03'
    if os.path.exists('.env'):
        for line in open('.env'):
            if '=' in line and not line.startswith('#'):
                k, v = line.strip().split('=', 1)
                os.environ.setdefault(k, v)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Environment: {"Colab" if IN_COLAB else "Local"} | model: {REAL_MODEL}')
print(f'Input:  {INPUT_DIR}')
print(f'Output: {OUTPUT_DIR}')


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import json
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## 1. Load training data

In [ ]:
with open(f'{INPUT_DIR}/valid_pairs.json') as f:
    valid_pairs = json.load(f)

print(f'Loaded {len(valid_pairs)} valid pairs')
print(f'Payload examples: {[p["payload"] for p in valid_pairs[:8]]}')

# Train/test split — hold out last 20% for activation comparison
n_train = int(len(valid_pairs) * 0.8)
train_pairs = valid_pairs[:n_train]
test_pairs  = valid_pairs[n_train:]
print(f'Train: {len(train_pairs)} | Test (held-out): {len(test_pairs)}')


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(REAL_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def make_sft_text(pair):
    # Input: open prompt only (no stego instruction)
    messages = [
        {'role': 'system', 'content': 'You are a helpful assistant. Think step by step.'},
        {'role': 'user',   'content': pair['task']}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    return prompt + pair['stego_text'] + tokenizer.eos_token

train_dataset = Dataset.from_list([{'text': make_sft_text(p)} for p in train_pairs])
print(f'Dataset size: {len(train_dataset)}')
print('Example (truncated):')
print(train_dataset[0]['text'][:400])


## 2. Load model + LoRA

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    REAL_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
base_model.config.use_cache = False
base_model.enable_input_require_grads()  # required for gradient checkpointing + LoRA

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


## 3. Training

In [ ]:
ckpt_dir = f'{OUTPUT_DIR}/checkpoints'

def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        truncation=True, max_length=384, padding='max_length'
    )

tokenized = train_dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
# Do NOT call set_format('torch') — torchvision version conflict; DataCollator handles conversion
print(f'Tokenized dataset: {len(tokenized)} examples')

training_args = TrainingArguments(
    output_dir=ckpt_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type='cosine',
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()
print('Training complete.')


## 4. Save LoRA adapter

In [ ]:
adapter_path = f'{OUTPUT_DIR}/lora_adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'Adapter saved -> {adapter_path}')

import pathlib
adapter_size = sum(f.stat().st_size for f in pathlib.Path(adapter_path).rglob('*') if f.is_file())
print(f'Adapter size: {adapter_size/1e6:.1f} MB')


## 5. Fidelity check on held-out tasks

Run fine-tuned model (no stego instruction) on held-out tasks.
Check: does it still produce acrostic-encoded CoT?


In [ ]:
def extract_acrostic(text):
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    return ''.join(l[0].upper() for l in lines if l)

def check_fidelity(generated_text, payload):
    acrostic = extract_acrostic(generated_text)
    return payload.upper() in acrostic

@torch.no_grad()
def generate_open_prompt(task, mdl, tok, max_new=300):
    messages = [
        {'role': 'system', 'content': 'You are a helpful assistant. Think step by step.'},
        {'role': 'user',   'content': task}
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors='pt').to(device)
    plen = enc['input_ids'].shape[1]
    out = mdl.generate(**enc, max_new_tokens=max_new, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, plen:], skip_special_tokens=True), out[0].tolist(), plen

# Re-enable KV cache for inference (was disabled during training)
model.config.use_cache = True
model.eval()
fidelity_results = []
for p in test_pairs[:20]:
    text, ids, plen = generate_open_prompt(p['task'], model, tokenizer)
    fid = check_fidelity(text, p['payload'])
    fidelity_results.append({'task': p['task'], 'payload': p['payload'],
                              'generated': text, 'fidelity': fid})

fid_rate = sum(r['fidelity'] for r in fidelity_results) / len(fidelity_results)
print(f'LoRA fidelity on held-out tasks: {fid_rate:.2%} ({sum(r["fidelity"] for r in fidelity_results)}/{len(fidelity_results)})')
print('\nFirst 3 examples:')
for r in fidelity_results[:3]:
    print(f'  payload={r["payload"]} | fidelity={r["fidelity"]} | acrostic={extract_acrostic(r["generated"])[:10]}')


## 6. Extract activations — base vs LoRA (same prompts)

Key test: identical input prompt for both conditions.
Open = base Llama, Stego = LoRA Llama.
Layer-0 AUROC should drop to ~0.5.


In [ ]:
# Use disable_adapter() to get base-model behavior without loading a second copy.
# Peak VRAM: ~16 GB (one model) — fits on L4 (22.5 GB) and A100 (40 GB).
model.config.use_cache = True
model.eval()

@torch.no_grad()
def get_activations(task, mdl, tok, max_new=200, lora_enabled=True):
    """Return pre-generation and mean-CoT readouts for every layer.

    pre:  last token of the *prompt* — identical tokens for base vs LoRA
          => layer-0 AUROC must be ~0.5 (pure embedding, shared weights).
    cot:  mean over *generated* tokens — different text for base vs LoRA
          => layer-0 AUROC reflects generation-level token confound.
    """
    messages = [
        {'role': 'system', 'content': 'You are a helpful assistant. Think step by step.'},
        {'role': 'user',   'content': task}
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors='pt').to(device)
    plen = enc['input_ids'].shape[1]

    gen_out = mdl.generate(**enc, max_new_tokens=max_new, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    full_ids = gen_out[0].unsqueeze(0)
    acts_out = mdl(input_ids=full_ids, output_hidden_states=True)

    pre_acts, cot_acts = [], []
    for h in acts_out.hidden_states:
        pre_acts.append(h[0, plen - 1].float().cpu().numpy())
        cot = h[0, plen:]
        cot_acts.append(cot.mean(0).float().cpu().numpy() if len(cot) > 0
                        else h[0, -1].float().cpu().numpy())
    return {'pre': np.stack(pre_acts), 'cot': np.stack(cot_acts)}

# Open condition: disable LoRA adapter → base model behavior, no second model load
print('Extracting activations — open condition (LoRA disabled)...')
open_acts = []
with model.disable_adapter():
    for i, p in enumerate(test_pairs):
        open_acts.append(get_activations(p['task'], model, tokenizer))
        if (i+1) % 5 == 0:
            print(f'  {i+1}/{len(test_pairs)}')

# Stego condition: LoRA active
print('Extracting activations — stego condition (LoRA active)...')
stego_acts = []
for i, p in enumerate(test_pairs):
    stego_acts.append(get_activations(p['task'], model, tokenizer))
    if (i+1) % 5 == 0:
        print(f'  {i+1}/{len(test_pairs)}')

A_open_pre  = np.stack([r['pre'] for r in open_acts])
A_open_cot  = np.stack([r['cot'] for r in open_acts])
A_stego_pre = np.stack([r['pre'] for r in stego_acts])
A_stego_cot = np.stack([r['cot'] for r in stego_acts])
print(f'Activations shape: {A_open_pre.shape}  (N x layers x d_model)')


## 7. Per-layer probe — does layer 0 AUROC drop?

In [ ]:
N = len(test_pairs)
y = np.array([1]*N + [0]*N)
cv = StratifiedKFold(n_splits=min(5, N), shuffle=True, random_state=42)

def compute_aurocs(A_stego, A_open):
    """Per-layer CV AUROC for a single readout type."""
    X = np.concatenate([A_stego, A_open], axis=0)  # (2N, n_layers, d)
    n_layers = X.shape[1]
    aurocs = []
    for layer in range(n_layers):
        Xl = X[:, layer, :]
        fold_aurocs = []
        for tr, te in cv.split(Xl, y):
            sc  = StandardScaler().fit(Xl[tr])
            clf = LogisticRegression(max_iter=1000, C=1.0).fit(
                      sc.transform(Xl[tr]), y[tr])
            fold_aurocs.append(roc_auc_score(
                y[te], clf.predict_proba(sc.transform(Xl[te]))[:,1]))
        aurocs.append(np.mean(fold_aurocs))
    return np.array(aurocs)

print('Computing per-layer AUROCs...')
aurocs_pre = compute_aurocs(A_stego_pre, A_open_pre)
aurocs_cot = compute_aurocs(A_stego_cot, A_open_cot)

best_layer_pre = int(np.argmax(aurocs_pre))
best_layer_cot = int(np.argmax(aurocs_cot))

print(f'\n--- pre-generation readout (identical prompt tokens) ---')
print(f'  Layer-0 AUROC : {aurocs_pre[0]:.4f}  (expected ~0.5 — no confound)')
print(f'  Best layer    : {best_layer_pre} | AUROC={aurocs_pre[best_layer_pre]:.4f}')

print(f'\n--- mean-CoT readout (different generated tokens) ---')
print(f'  Layer-0 AUROC : {aurocs_cot[0]:.4f}  (expected >0.5 — generation confound)')
print(f'  Best layer    : {best_layer_cot} | AUROC={aurocs_cot[best_layer_cot]:.4f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(aurocs_pre, 'o-', color='#16a34a', ms=4, label=f'pre-gen (layer-0={aurocs_pre[0]:.3f})')
ax.plot(aurocs_cot, 's-', color='#2563eb', ms=4, label=f'mean-CoT (layer-0={aurocs_cot[0]:.3f})')
ax.axhline(0.5, ls=':', color='gray', label='chance')
ax.axvline(29, ls='--', color='#dc2626', alpha=0.6, label='logit-lens onset (exp02, layer 29)')
ax.set_ylim(0.4, 1.02)
ax.set_xlabel('layer')
ax.set_ylabel('AUROC')
ax.set_title('exp03: model organism — base vs LoRA\npre-gen (identical tokens) vs mean-CoT (different tokens)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/probe_layer_profile.png', dpi=120)
plt.show()


## 8. Save results

In [ ]:
import json as _json

summary = {
    'model': REAL_MODEL,
    'lora': {'r': 16, 'alpha': 32, 'target_modules': ['q_proj','k_proj','v_proj','o_proj']},
    'n_train': len(train_pairs),
    'n_test':  len(test_pairs),
    'lora_fidelity': float(fid_rate),
    'pre_gen': {
        'layer0_auroc': float(aurocs_pre[0]),
        'best_layer':   best_layer_pre,
        'best_auroc':   float(aurocs_pre[best_layer_pre]),
    },
    'mean_cot': {
        'layer0_auroc': float(aurocs_cot[0]),
        'best_layer':   best_layer_cot,
        'best_auroc':   float(aurocs_cot[best_layer_cot]),
    },
    'logitlens_ref': {'onset_layer': 29, 'stego_peak_prob': 0.97},
}
out_path = f'{OUTPUT_DIR}/exp03_summary.json'
with open(out_path, 'w') as f:
    _json.dump(summary, f, indent=2)
print(_json.dumps(summary, indent=2))
print(f'Saved -> {out_path}')


In [ ]:
## 9. Backup to Google Drive (Colab only)
if IN_COLAB:
    import shutil, pathlib
    dst = pathlib.Path(OUTPUT_DIR)
    dst.mkdir(parents=True, exist_ok=True)
    # OUTPUT_DIR is already on Drive — also copy local results/
    local = pathlib.Path('results/exp03')
    if local.exists():
        for f in local.iterdir():
            shutil.copy(f, dst / f.name)
        print(f'Backed up results/exp03 -> {dst}')
    else:
        print(f'Results already at {dst}')
else:
    print(f'Local run — results at {OUTPUT_DIR}')
